# Memory AI Lab — Évaluation ARI V3

**Workflow :** éditer le code dans VS Code → `git push` → ouvrir ce notebook dans [colab.research.google.com](https://colab.research.google.com)

**GPU requis** → `Exécution > Modifier le type d'exécution > GPU T4`

## Protocole anti-surapprentissage

```
group_gold_tune.json  → 448 épisodes, 8954 msgs  (août 2023 → jan 2025)
                         ↑ Optuna bayésien — 60 trials
                           Seuls attach_threshold, ema_alpha, time_threshold sont tunés.
                           hard_break et dormancy = valeurs sémantiques fixes.

group_gold_test.json  → 193 épisodes, 2786 msgs  (jan 2025 → mars 2026)
                         ↑ SCORE FINAL — une seule fois, ne pas tuner dessus
```

**Données requises sur Google Drive (`memory_ai_data/`) :**
```
group_anon.txt
group_gold_tune.json
group_gold_test.json
```

In [ ]:
# ── CELLULE 1 : Code depuis GitHub ────────────────────────────────────────
import os, sys
REPO = 'https://github.com/Eloekamaje/memory_ai.git'
CODE_DIR = '/content/memory_ai'
if os.path.exists(CODE_DIR):
    !git -C {CODE_DIR} pull --quiet
else:
    !git clone {REPO} {CODE_DIR} --quiet
sys.path.insert(0, f'{CODE_DIR}/src')
print('✓ Code prêt')

In [ ]:
# ── CELLULE 2 : Dépendances ────────────────────────────────────────────────
# Fix sympy/torch incompatibilité — DOIT être installé avant tout import torch
!pip install "sympy==1.13.1" -q
!pip install -r {CODE_DIR}/requirements_colab.txt -q
!pip install optuna -q
!python -m spacy download fr_core_news_sm -q
print('✓ OK')
print()
print('⚠️  Si première exécution : Exécution > Redémarrer la session,')
print('   puis relancer à partir de la cellule 3 (les imports sont en cache).')

In [ ]:
# ── CELLULE 4 : Parse + Embeddings complets (GPU + cache) ─────────────────
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from parsers.whatsapp_parser import parse_whatsapp_chat

EMBED_CACHE = Path(DATA_DIR) / 'group_embeddings.npy'
all_artifacts = parse_whatsapp_chat(f'{DATA_DIR}/group_anon.txt')
texts = [a.content for a in all_artifacts]
print(f'[1/2] {len(texts)} messages parsés')

if EMBED_CACHE.exists():
    all_embeddings = np.load(EMBED_CACHE)
    assert len(all_embeddings) == len(texts), 'Cache périmé — supprimer group_embeddings.npy'
    print('[2/2] Embeddings chargés depuis cache')
else:
    print(f'[2/2] Calcul sur {device} ...')
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
    all_embeddings = model.encode(texts, batch_size=256, show_progress_bar=True,
                                  device=device, convert_to_numpy=True).astype(np.float32)
    np.save(EMBED_CACHE, all_embeddings)
    print(f'      Sauvegardé → {EMBED_CACHE}')
print(f'      Shape : {all_embeddings.shape}')

In [ ]:
# ── CELLULE 5 : Charger tune / test ───────────────────────────────────────
import json

def load_split(path):
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    n = len(data['artifacts'])
    y_true = [None] * n
    for ep in data['episodes']:
        for idx in range(ep['start_idx'], ep['end_idx'] + 1):
            if idx < n:
                y_true[idx] = ep['episode_id']
    return data['artifacts'], y_true, data['episodes'], data['meta']

tune_arts, y_true_tune, tune_eps, tune_meta = load_split(f'{DATA_DIR}/group_gold_tune.json')
test_arts, y_true_test, test_eps, test_meta = load_split(f'{DATA_DIR}/group_gold_test.json')

n_tune = len(tune_arts)
arts_tune = all_artifacts[:n_tune]
arts_test = all_artifacts[n_tune:n_tune + len(test_arts)]
emb_tune  = all_embeddings[:n_tune]
emb_test  = all_embeddings[n_tune:n_tune + len(test_arts)]

print(f'✓ Tune : {len(tune_eps)} épisodes · {n_tune} msgs · {tune_meta["period"]}')
print(f'✓ Test : {len(test_eps)} épisodes · {len(test_arts)} msgs · {test_meta["period"]}')

In [ ]:
# ── CELLULE 6 : Helper d'évaluation ───────────────────────────────────────
from episode_algorithm_fast import EpisodeSegmenterFast
from episode_splitter_fast import EpisodeSplitterFast, SplitConfig
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')

SPLITTER = EpisodeSplitterFast(SplitConfig(
    min_cohesion=0.65, min_size_to_split=8,
    max_span_hours=168.0, max_splits=6,
    min_sub_size=3, silhouette_threshold=0.10,
), device=DEVICE)

FIXED_PARAMS = dict(
    alpha=0.45, beta=0.25, gamma=0.10, delta=0.20, rho=0.05,
    dormancy_minutes=1440,
    hard_break_minutes=0,
    active_penalty_hours=24.0,
    allow_reactivation=True,
)

def run_eval(artifacts, embeddings, y_true, tunable_params, verbose=False, use_splitter=False):
    seg = EpisodeSegmenterFast(device=DEVICE, **{**FIXED_PARAMS, **tunable_params})
    eps = seg.consolidate(seg.segment(artifacts, embeddings))
    if use_splitter:
        eps = SPLITTER.split(eps, artifacts, embeddings, verbose=False)
    n = len(artifacts)
    y_pred = [None] * n
    for ep in eps:
        for idx in ep.artifact_indices:
            if idx < n: y_pred[idx] = ep.id
    pairs = [(t, p) for t, p in zip(y_true, y_pred) if t is not None and p is not None]
    if not pairs:
        return 0.0, 0.0, len(eps)
    yt, yp = zip(*pairs)
    ari = adjusted_rand_score(yt, yp)
    nmi = normalized_mutual_info_score(yt, yp)
    if verbose:
        n_gold = len(set(t for t in y_true if t is not None))
        print(f'  ARI={ari:+.4f}  NMI={nmi:.4f}  gold={n_gold}  pred={len(eps)}')
    return ari, nmi, len(eps)

print('✓ Helpers prêts (EpisodeSegmenterFast + EpisodeSplitterFast)')

In [ ]:
# ── CELLULE 7 : Optimisation bayésienne (Optuna + CMA-ES) sur TUNE ─────────
import importlib, subprocess, sys
if importlib.util.find_spec('optuna') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'])

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_FAST   = 1000
N_TRIALS = 20
N_JOBS   = 2

arts_fast = arts_tune[:N_FAST]
emb_fast  = emb_tune[:N_FAST]
y_fast    = y_true_tune[:N_FAST]
print(f'Tuning {N_TRIALS} trials × {N_FAST} msgs | n_jobs={N_JOBS} | device={DEVICE}')

# Phase 1 : QMC pour explorer l'espace uniformément (5 trials)
# Phase 2 : CMA-ES pour converger — modélise les corrélations entre params
# attach_threshold et ema_alpha interagissent → CMA-ES plus adapté que TPE
sampler = optuna.samplers.CmaEsSampler(
    seed=42,
    n_startup_trials=5,        # exploration QMC avant CMA-ES
    restart_strategy='ipop',   # redémarre si convergence prématurée
)

def objective(trial):
    params = dict(
        attach_threshold      = trial.suggest_float('attach',   0.25, 0.60),
        ema_alpha             = trial.suggest_float('ema',      0.50, 0.95),
        time_threshold_minutes= trial.suggest_int  ('time_thr', 60, 480, step=30),
    )
    ari, _, _ = run_eval(arts_fast, emb_fast, y_fast, params)
    return ari

study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=N_JOBS,
    callbacks=[lambda s, t: print(f'  Trial {t.number:3d} | ARI={t.value:+.4f} | {t.params}')],
)

raw = study.best_params
best_params = dict(
    attach_threshold      = raw['attach'],
    ema_alpha             = raw['ema'],
    time_threshold_minutes= raw['time_thr'],
)
print(f'\n✓ Meilleurs paramètres (CMA-ES) :')
for k, v in best_params.items():
    print(f'  {k} = {v:.4f}')
print(f'  ARI fast-tune = {study.best_value:+.4f}')

In [ ]:
# ── CELLULE 8 : Importance des paramètres ─────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

# Importance de chaque paramètre sur l'ARI
importance = optuna.importance.get_param_importances(study)
print('Importance des paramètres :')
for k, v in importance.items():
    bar = '█' * int(v * 40)
    print(f'  {k:25s} {bar} {v:.3f}')

# Historique des trials
df_trials = study.trials_dataframe()[['number','value','params_attach','params_ema','params_time_thr']]
df_trials.columns = ['trial', 'ari', 'attach', 'ema', 'time_thr']
print(f'\nTop 5 trials :')
print(df_trials.sort_values('ari', ascending=False).head(5).to_string(index=False))

In [ ]:
# ── CELLULE 9 : Score FINAL sur TEST ──────────────────────────────────────
# Exécuter UNE SEULE FOIS avec les params optuna
# Ne pas re-run après avoir vu le score

# Mapper les noms courts Optuna → noms réels EpisodeSegmenter
raw = study.best_params
best_params = dict(
    attach_threshold      = raw['attach'],
    ema_alpha             = raw['ema'],
    time_threshold_minutes= raw['time_thr'],
)

print('Évaluation TUNE (référence) :')
ari_tune, nmi_tune, n_pred_tune = run_eval(arts_tune, emb_tune, y_true_tune, best_params, verbose=True)

print('\nÉvaluation TEST (officiel) :')
ari_test, nmi_test, n_pred_test = run_eval(arts_test, emb_test, y_true_test, best_params, verbose=True)

gap = ari_tune - ari_test
gap_status = '✓ OK — bonne généralisation' if abs(gap) < 0.05 else '⚠️  Écart élevé — vérifier'

print(f"""
╔══════════════════════════════════════════════════╗
║  RÉSULTAT OFFICIEL V3                            ║
╠══════════════════════════════════════════════════╣
║  ARI  (test)  : {ari_test:+.4f}                  ║
║  NMI  (test)  : {nmi_test:.4f}                   ║
║  Gold (test)  : {len(test_eps)} épisodes          ║
║  Pred (test)  : {n_pred_test} épisodes            ║
╠══════════════════════════════════════════════════╣
║  ARI  (tune)  : {ari_tune:+.4f}                  ║
║  Gap t-t      : {gap:+.4f}  {gap_status}  ║
╚══════════════════════════════════════════════════╝
""")
print('Paramètres finaux :')
for k, v in {**FIXED_PARAMS, **best_params}.items():
    print(f'  {k} = {v}')